In [ ]:
# MIT Lab 3 | LLM Fine-tuning
# Base model: Liquid AI LFM2-1.2B
# Custom dataset: Drake/rap lyrics (fine-tune to respond in lyrics style)

!pip install transformers datasets peft accelerate bitsandbytes --quiet

import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00
PyTorch: 2.10.0+cu128
GPU: True
GPU name: Tesla T4


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"Model: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model: gpt2
Parameters: 124,439,808


In [ ]:
from datasets import load_dataset

# We'll use a subset of the Alpaca dataset for quick experimentation
dataset_name = "yahma/alpaca-cleaned"
dataset = load_dataset(dataset_name, split="train[:5000]")

print(f"Loaded dataset: {dataset_name}")
print(f"Number of training examples: {len(dataset)}")
print("\nLook at a sample example:")
print(dataset[0])

README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Loaded dataset: yahma/alpaca-cleaned
Number of training examples: 5000

Look at a sample example:
{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'input': '', 'instruction': 'Give three tips for staying healthy.'}


In [ ]:
def format_and_tokenize(example):
    # Combine the data into a single prompt format
    if example.get("input", "") != "":
        prompt = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}{tokenizer.eos_token}"
    else:
        prompt = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}{tokenizer.eos_token}"

    # Tokenize the prompt string
    tokens = tokenizer(
        prompt,
        truncation=True,
        max_length=256,
        padding="max_length"
    )

    # In CausalLM, the labels are the same as the input_ids (the model shifts them internally to predict the next token)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

# Apply the function to our whole dataset
tokenized_dataset = dataset.map(format_and_tokenize, remove_columns=dataset.column_names)

print("Dataset successfully tokenized and ready for training!")

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset successfully tokenized and ready for training!


In [ ]:
from peft import LoraConfig, get_peft_model

# 1. Define the LoRA configuration
lora_config = LoraConfig(
    r=8,               # Rank of the update matrices (lower = faster, higher = more expressive)
    lora_alpha=32,     # Scaling factor
    target_modules=["c_attn"], # Target GPT-2's attention blocks
    lora_dropout=0.05, # Prevent overfitting
    bias="none",
    task_type="CAUSAL_LM"
)

# 2. Wrap our base GPT-2 model with the LoRA config
peft_model = get_peft_model(model, lora_config)

# 3. Check how many parameters we are actually training
peft_model.print_trainable_parameters()

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# 1. Set up the hyper-parameters for training
training_args = TrainingArguments(
    output_dir="./gpt2-alpaca-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4, # Effectively gives us a batch size of 16 (4x4)
    learning_rate=2e-4,
    max_steps=100,                 # Short run for practice
    logging_steps=10,              # Print loss every 10 steps
    fp16=True,                     # Use half-precision to save memory
    optim="adamw_torch",
    report_to="none"               # Disables wandb/tensorboard logging for this quick test
)

# 2. The Data Collator automatically pads our batches during training
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 3. Initialize the Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Trainer setup complete! Ready for liftoff.")

Trainer setup complete! Ready for liftoff.


In [ ]:
import time

print("Starting training...")
start_time = time.time()

# This is the actual fine-tuning process
trainer.train()

end_time = time.time()
print(f"Training completed in {(end_time - start_time) / 60:.2f} minutes.")

# Save the LoRA adapters (not the whole model, just the 'delta' we trained)
peft_model.save_pretrained("./gpt2-alpaca-lora-adapter")

Starting training...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.061522
20,2.885078
30,2.799043
40,2.741280
50,2.621426
60,2.548883
70,2.580514
80,2.502953
90,2.507118
100,2.508540


Training completed in 0.95 minutes.


In [ ]:
# 1. Define a test instruction
instruction = "Give me a creative title for a story about a robot learning to paint."
prompt = f"### Instruction:\n{instruction}\n\n### Response:"

# 2. Tokenize and move to GPU
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 3. Generate a response
peft_model.eval() # Set to evaluation mode
with torch.no_grad():
    outputs = peft_model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# 4. Decode and print
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("-" * 30)
print(response)
print("-" * 30)

------------------------------
### Instruction:
Give me a creative title for a story about a robot learning to paint.

### Response:
This story is about a robot learning to paint. The robot is an electrical engineer, who has taken on a hobby and is now learning to paint. The robot starts by learning to paint using a brush, then moves on to a different brush and
------------------------------


In [ ]:
# Let's override the previous training args for a more aggressive run
training_args.max_steps = 300      # Triple the training time
training_args.learning_rate = 3e-4 # Slightly higher learning rate
training_args.logging_steps = 20

# Re-initialize and train again (this will continue from the current state)
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Starting high-intensity fine-tuning...")
trainer.train()
print("Done! Now go back and run 'Cell 8' again to see if it stopped talking about animals.")

Starting high-intensity fine-tuning...


Step,Training Loss
20,2.434623
40,2.395664
60,2.315494
80,2.313226
100,2.301004
120,2.327121
140,2.334143
160,2.338895
180,2.277827
200,2.290765


Done! Now go back and run 'Cell 8' again to see if it stopped talking about animals.


In [ ]:
!pip install lion-pytorch --quiet

In [ ]:
import torch.nn.functional as F

def forward_and_compute_loss(model, tokens, mask, context_length=256):
    tokens = tokens[:, :context_length]
    mask = mask[:, :context_length]

    # Shift inputs and targets for Causal LM
    x = tokens[:, :-1]
    y = tokens[:, 1:]
    mask = mask[:, 1:]

    logits = model(x).logits

    # Calculate loss for every token in the sequence
    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        y.reshape(-1),
        reduction="none"
    )

    # MIT Lab Secret: Only calculate loss on the "Response" tokens (the mask)
    masked_loss = loss[mask.reshape(-1)].mean()
    return masked_loss

In [ ]:
from lion_pytorch import Lion

# Setup the optimizer
optimizer = Lion(peft_model.parameters(), lr=1e-4)

peft_model.train()
print("Starting manual fine-tuning...")

for step in range(50): # Run 50 steps to see it in action
    # Get a batch from your existing tokenized_dataset
    batch = tokenized_dataset[step % len(tokenized_dataset)]

    input_ids = torch.tensor([batch["input_ids"]]).to("cuda")
    # Using a full mask for now; in the lab, you'd refine this to only the answer
    mask = torch.ones_like(input_ids, dtype=torch.bool).to("cuda")

    # Forward, Backward, Step
    loss = forward_and_compute_loss(peft_model, input_ids, mask)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 10 == 0:
        print(f"Step {step} | Loss: {loss.item():.4f}")

print("Manual training complete!")

Starting manual fine-tuning...
Step 0 | Loss: 5.9966
Step 10 | Loss: 5.4234
Step 20 | Loss: 2.1958
Step 30 | Loss: 0.4391
Step 40 | Loss: 1.4162
Manual training complete!


In [ ]:
# Test the 'Yoda' or 'Robot' style again
instruction = "Explain why the sky is blue."
prompt = f"### Instruction:\n{instruction}\n\n### Response:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
peft_model.eval()

with torch.no_grad():
    outputs = peft_model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.5, # Lower temperature for more stability
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

print("-" * 30)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("-" * 30)

------------------------------
### Instruction:
Explain why the sky is blue.

### Response:
------------------------------
